# Tutorial 11 GANs: Generation and Image Translation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JimengShi/SysEng_Cornell/blob/main/Tutorial_11_Generative_Modeling_GAN/tutorial_11_pytorch.ipynb)

How do two competing networks learn to generate and translate images? Train a compact DCGAN, explore its latent space, then compare paired pix2pix-style and unpaired CycleGAN-style objectives.

**Start:** Open in Colab → connect → **Runtime → Run all**, or use Shift+Enter to work cell by cell. Required package setup is included below. No local installation, API key, or Drive mount is needed.

**Prerequisites:** PyTorch tensors, backpropagation, CNNs, and Tutorial 10 concepts.

**Resources:** Default: CPU, internet, MNIST download (~12 MB), and several minutes. No pretrained weights or paid service. Larger mode increases data and epochs; a GPU is helpful but not required by the code.

Each notebook is independent: earlier lessons are helpful but their runtime state and files are not required. While the GitHub repository is private, authorize Colab to access it, or download this notebook and use **File → Upload notebook** in Colab.

## Learning goals and scope

- Trace which parameters and gradients change in each adversarial phase.
- Separate generator loss, image appearance, and diversity: none alone establishes distribution matching.
- Visualize fixed-noise training, interpolation, and latent inversion with a frozen generator.
- Contrast paired conditional adversarial + L1 learning with unpaired adversarial + cycle + identity learning.

This is a **compact teaching adaptation**, not a reproduction of the papers. DCGAN generates 28×28 digits. Paired translation maps digit edges to grayscale digits; unpaired translation uses disjoint sets of normal and inverted digits. These replace the upstream large facades and horse/zebra experiments. We use a real skip connection and patch discriminator, but narrower networks, shorter schedules, and no replay buffer. Cycle consistency does not guarantee semantic correctness. The upstream introduction's TF-Hub example is replaced by local PyTorch latent exploration; TensorFlow is not needed.

## 1. Environment

This cell installs missing packages inside the current runtime. It keeps an existing PyTorch/torchvision pair together and does not downgrade Colab’s installed stack. If imports fail after other experiments, start a fresh runtime and run from the top.

In [ ]:
import importlib.util
import subprocess
import sys

packages = {'torch': 'torch', 'torchvision': 'torchvision', 'numpy': 'numpy', 'pandas': 'pandas', 'matplotlib': 'matplotlib', 'PIL': 'pillow'}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if "torch" in missing or "torchvision" in missing:
    missing = sorted(set(missing + ["torch", "torchvision"]))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
else:
    print("Required packages are already installed.")

In [ ]:
import json, platform, random, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Dataset, Subset
import torchvision
from torchvision import datasets, transforms, models
from PIL import Image
from IPython.display import display
%matplotlib inline

SEED = 42
QUICK_RUN = True
DEVICE_PREFERENCE = "auto"  # "cpu" forces CPU; "auto" uses CUDA when available.
assert DEVICE_PREFERENCE in {"auto", "cpu"}
device = torch.device("cuda" if DEVICE_PREFERENCE == "auto" and torch.cuda.is_available() else "cpu")
def seed_everything():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
seed_everything()
torch.set_num_threads(min(2, torch.get_num_threads()))
DATA_DIR = Path("data") / "tutorial_11"
OUTPUT_DIR = Path("outputs") / "tutorial_11"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})
versions = {"python": platform.python_version(), "torch": str(torch.__version__),
            "torchvision": str(torchvision.__version__), "numpy": np.__version__,
            "pandas": pd.__version__, "matplotlib": matplotlib.__version__,
            "device": str(device), "seed": SEED, "quick_run": QUICK_RUN}
display(pd.Series(versions, name="Environment"))
run_started = time.perf_counter()

from IPython.display import Image as DisplayImage
from torchvision.utils import make_grid
import copy, itertools
FIGURES = []
def finish(fig, name):
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / name, bbox_inches="tight")
    FIGURES.append(name)
    plt.show()
def panel(rows, labels, name, title):
    fig, axes = plt.subplots(len(rows), len(rows[0]), figsize=(12, 1.8*len(rows)), squeeze=False)
    for r, (batch, label) in enumerate(zip(rows, labels)):
        for j, im in enumerate(batch):
            axes[r,j].imshow(im.detach().cpu().squeeze(), cmap="gray", vmin=-1, vmax=1)
            axes[r,j].set_xticks([]); axes[r,j].set_yticks([])
        axes[r,0].set_ylabel(label)
    fig.suptitle(title); finish(fig, name)
def frozen(model, flag=True):
    for p in model.parameters(): p.requires_grad_(not flag)
def snapshot(model): return {k:v.detach().clone() for k,v in model.state_dict().items()}
def unchanged(model, before): return all(torch.equal(v,before[k]) for k,v in model.state_dict().items())
def finite(*losses):
    assert all(torch.isfinite(x).all() for x in losses), "Non-finite loss"

## 2. Data, range, and reproducible splits

Use a balanced random subset of the official MNIST training split; reserve separate validation rows and the official test split. Normalize grayscale pixels from [0,255] to **[-1,1]**, matching the generator's Tanh. Labels only balance the samples; these models do not condition on class labels. Test images are never used for training or model selection. Validation selects the paired translator by L1; DCGAN and CycleGAN use a predeclared final epoch because their losses are not reliable quality rankings.

In [ ]:
raw = datasets.MNIST(DATA_DIR, train=True, download=True)
test_raw = datasets.MNIST(DATA_DIR, train=False, download=True)
def balanced_indices(labels, per_class, seed):
    rng = torch.Generator().manual_seed(seed)
    return torch.cat([torch.where(labels==k)[0][torch.randperm(int((labels==k).sum()), generator=rng)[:per_class]] for k in range(10)])
all_indices = balanced_indices(raw.targets, 700 if QUICK_RUN else 5000, SEED)
per_class = len(all_indices)//10
train_idx = torch.cat([all_indices[k*per_class:k*per_class+per_class-(100 if QUICK_RUN else 400)] for k in range(10)])
val_idx = torch.tensor(sorted(set(all_indices.tolist())-set(train_idx.tolist())))
test_idx = balanced_indices(test_raw.targets, 100, SEED+1)
assert not set(train_idx.tolist()) & set(val_idx.tolist())
def pixels(ds, ids): return ds.data[ids].float().unsqueeze(1)/127.5-1
train_x, val_x, test_x = pixels(raw,train_idx), pixels(raw,val_idx), pixels(test_raw,test_idx)
def loader(x, batch_size=64, seed=SEED):
    return DataLoader(TensorDataset(x), batch_size=batch_size, shuffle=True, num_workers=0,
                      generator=torch.Generator().manual_seed(seed))
# Fixed randomly selected examples, independent of model outcomes.
view_ids = torch.randperm(len(test_x), generator=torch.Generator().manual_seed(SEED+2))[:8]
examples = test_x[view_ids]
assert train_x.min()>=-1 and train_x.max()<=1
panel([examples], ["Real test"], "real_digits.png", "MNIST: fixed held-out examples")
print("Train / validation / test:", len(train_x), len(val_x), len(test_x))

## 3. DCGAN: noise → image, image → real/fake logit

The generator maps `[B,32] → [B,32,7,7] → [B,16,14,14] → [B,1,28,28]`. The discriminator returns one **raw logit** per image. `BCEWithLogitsLoss` combines sigmoid and cross-entropy stably; do not apply a second sigmoid inside the loss.

The generator uses batch normalization; D has neither batch normalization nor dropout. During the D step, G runs in evaluation mode so its running statistics stay fixed. During the G step it returns to training mode. This is a narrow 28×28 adaptation rather than the canonical DCGAN architecture.

For discriminator logits d and generator G:

- D loss = BCE(D(real),1) + BCE(D(G(z).detach()),0).
- Non-saturating G loss = BCE(D(G(z)),1).

During D's update, fake pixels are detached from G. During G's update, D parameters are frozen **without** `no_grad`: gradients must still pass through D to G. A discriminator score of 0.5 can also indicate an undertrained discriminator; it is not a certificate of realism.

In [ ]:
LATENT = 32
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(LATENT,32*7*7),nn.Unflatten(1,(32,7,7)),nn.BatchNorm2d(32),nn.ReLU(),
             nn.ConvTranspose2d(32,16,4,2,1),nn.BatchNorm2d(16),nn.ReLU(),nn.ConvTranspose2d(16,1,4,2,1),nn.Tanh())
    def forward(self,z): return self.net(z)
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(nn.Conv2d(1,16,4,2,1),nn.LeakyReLU(.2),
             nn.Conv2d(16,32,4,2,1),nn.LeakyReLU(.2),nn.Flatten(),nn.Linear(32*7*7,1))
    def forward(self,x): return self.net(x)
G,D=Generator().to(device),Discriminator().to(device)
opt_g=torch.optim.Adam(G.parameters(),lr=.0002,betas=(.5,.999))
opt_d=torch.optim.Adam(D.parameters(),lr=.0002,betas=(.5,.999))
bce=nn.BCEWithLogitsLoss()
def real_loss(logits): return bce(logits,torch.ones_like(logits))
def fake_loss(logits): return bce(logits,torch.zeros_like(logits))
fixed_z=torch.randn(16,LATENT,generator=torch.Generator().manual_seed(SEED+3)).to(device)
assert G(fixed_z).shape==(16,1,28,28) and D(G(fixed_z)).shape==(16,1)
assert torch.allclose(real_loss(torch.zeros(4,1)),torch.tensor(np.log(2),dtype=torch.float32))
checks={"shape_and_logit_loss": True, "disjoint_splits": True}
def gan_step(real, audit=False):
    frozen(D,False); opt_d.zero_grad(set_to_none=True); opt_g.zero_grad(set_to_none=True)
    if audit: g0,d0=snapshot(G),snapshot(D)
    z=torch.randn(len(real),LATENT,device=device)
    G.eval()  # Freeze batch-normalization running statistics during the D-only phase.
    with torch.no_grad(): fake=G(z)
    ld=real_loss(D(real))+fake_loss(D(fake)); finite(ld); ld.backward()
    if audit: assert all(p.grad is None for p in G.parameters())
    opt_d.step()
    if audit: assert unchanged(G,g0) and not unchanged(D,d0)
    frozen(D); opt_d.zero_grad(set_to_none=True); opt_g.zero_grad(set_to_none=True)
    if audit: d1=snapshot(D)
    G.train()
    lg=real_loss(D(G(torch.randn(len(real),LATENT,device=device))))
    finite(lg); lg.backward()
    if audit:
        assert all(p.grad is None for p in D.parameters())
        assert sum(p.grad.abs().sum().item() for p in G.parameters())>0
    opt_g.step()
    if audit:
        assert unchanged(D,d1) and not unchanged(G,g0)
        checks["gan_gradient_and_update_isolation"]=True
    frozen(D,False)
    return ld.item(),lg.item()

## 4. Train and watch the same noise across epochs

Quick mode uses 6,000 training images and twelve epochs. Larger mode uses 46,000 images and 25 epochs. We save the **final epoch**, not the epoch with the lowest G loss. Every animation frame uses the same 16 noise vectors, with fixed display scaling; no favorable examples are selected. Short runs may show indistinct digits, checkerboard artifacts, or collapse.

In [ ]:
GAN_EPOCHS=12 if QUICK_RUN else 25
def frame():
    G.eval()
    with torch.no_grad(): grid=make_grid((G(fixed_z).cpu()+1)/2,nrow=4,padding=2)
    G.train()
    return Image.fromarray((grid.permute(1,2,0).numpy().clip(0,1)*255).astype("uint8"))
frames=[frame()]; gan_history=[]
for epoch in range(1,GAN_EPOCHS+1):
    sums=np.zeros(2); count=0
    for batch_index,(x,) in enumerate(loader(train_x,seed=SEED+epoch)):
        losses=gan_step(x.to(device),audit=epoch==1 and batch_index==0)
        sums+=np.array(losses)*len(x); count+=len(x)
    gan_history.append({"epoch":epoch,"D":float(sums[0]/count),"G":float(sums[1]/count)})
    frames.append(frame()); print("DCGAN",gan_history[-1])
G.eval(); D.eval()
frames[0].save(OUTPUT_DIR/"gan_training.gif",save_all=True,append_images=frames[1:],duration=600,loop=0)
with Image.open(OUTPUT_DIR/"gan_training.gif") as gif:
    assert gif.n_frames==GAN_EPOCHS+1
checks["gif_roundtrip"]=True
with torch.no_grad(): generated=G(fixed_z).cpu()
panel([generated[:8],generated[8:]], ["Fixed noise 1–8","Fixed noise 9–16"],"gan_samples.png","Final-epoch generated samples (not reconstructions)")
display(DisplayImage(filename=str(OUTPUT_DIR/"gan_training.gif")))
fig,ax=plt.subplots(figsize=(7,3)); pd.DataFrame(gan_history).set_index("epoch").plot(ax=ax)
ax.set(xlabel="Epoch",ylabel="Binary cross-entropy",title="Adversarial losses are not image-quality scores")
finish(fig,"gan_losses.png")

## 5. Latent interpolation and inversion

Interpolation uses `(1−t)z₀ + tz₁`; intermediate vectors do not generally have the same distribution as the Gaussian prior. Smooth changes alone do not establish semantic organization.

For inversion, freeze G and optimize only a new z to reduce pixel MSE against the **first fixed test digit**. Save the best z encountered, including initialization. This is per-image fitting, not retraining G or evaluating an encoder. Even perfect optimization cannot recover a target outside the generator's range, and a matching image does not establish recovery of a unique latent code.

In [ ]:
frozen(G); g_before=snapshot(G)
t=torch.linspace(0,1,8,device=device).unsqueeze(1)
with torch.no_grad(): interpolated=G((1-t)*fixed_z[:1]+t*fixed_z[1:2]).cpu()
panel([interpolated],["t: 0 → 1"],"latent_interpolation.png","Linear path between two fixed prior vectors")
target=examples[:1].to(device)
z=torch.randn(1,LATENT,device=device,requires_grad=True)
z_optimizer=torch.optim.Adam([z],lr=.05)
with torch.no_grad(): initial=G(z).cpu(); best_mse=F.mse_loss(G(z),target).item()
best_z=z.detach().clone(); inversion=[best_mse]
for step in range(100 if QUICK_RUN else 500):
    z_optimizer.zero_grad(); loss=F.mse_loss(G(z),target); finite(loss); loss.backward()
    assert z.grad is not None and torch.isfinite(z.grad).all()
    z_optimizer.step()
    with torch.no_grad(): current=F.mse_loss(G(z),target).item()
    inversion.append(current)
    if current<best_mse: best_mse,best_z=current,z.detach().clone()
with torch.no_grad(): fitted=G(best_z).cpu()
assert unchanged(G,g_before) and best_mse<=inversion[0]
checks["inversion_frozen_generator"]=True
panel([torch.cat([examples[:1],initial,fitted])],["Target / initial / best"],"latent_inversion.png","Pixel fitting with a frozen GAN generator")
fig,ax=plt.subplots(figsize=(7,3));ax.plot(inversion)
ax.set(xlabel="Latent optimization step",ylabel="Pixel MSE on [-1,1]",title="Inversion of one held-out example; not a population score")
finish(fig,"inversion_loss.png")
print("Initial / best inversion MSE:",inversion[0],best_mse)

## 6. Paired translation: a small pix2pix-style model

Create edge inputs deterministically from the same target digit using local max–min differences. Unlike latent inversion, an image translator predicts directly from an input. Its U-Net concatenates a downsampling feature map with an upsampling feature map. The discriminator sees `[edge, target]` or `[edge, generated]` as two channels and returns a **patch logit map**, not a segmentation mask.

G loss = BCE(D(edge,G(edge)),1) + 50 × mean absolute error. D uses real and detached fake pairs. Both losses average over batch and patch/pixel dimensions. Validation L1 selects the model; a held-out test evaluates once. Compare against simply returning the edge input and the training mean image. These baselines do not isolate the benefit of adversarial training; an L1-only ablation is an exercise.

In [ ]:
def edges(x):
    unit=(x+1)/2
    return (F.max_pool2d(unit,3,1,1)+F.max_pool2d(-unit,3,1,1))*2-1
class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.down1=nn.Sequential(nn.Conv2d(1,16,4,2,1),nn.LeakyReLU(.2))
        self.down2=nn.Sequential(nn.Conv2d(16,32,4,2,1),nn.LeakyReLU(.2))
        self.up=nn.Sequential(nn.ConvTranspose2d(32,16,4,2,1),nn.ReLU())
        self.out=nn.Sequential(nn.ConvTranspose2d(32,1,4,2,1),nn.Tanh())
    def forward(self,x):
        skip=self.down1(x)
        return self.out(torch.cat([self.up(self.down2(skip)),skip],dim=1))
class PatchD(nn.Module):
    def __init__(self,channels=2):
        super().__init__()
        self.net=nn.Sequential(nn.Conv2d(channels,16,4,2,1),nn.LeakyReLU(.2),
            nn.Conv2d(16,32,4,2,1),nn.LeakyReLU(.2),nn.Conv2d(32,1,3,1,1))
    def forward(self,x): return self.net(x)
P,PD=UNet().to(device),PatchD().to(device)
pg=torch.optim.Adam(P.parameters(),lr=.001,betas=(.5,.999))
pd_opt=torch.optim.Adam(PD.parameters(),lr=.0002,betas=(.5,.999))
paired_ids=torch.randperm(len(train_x),generator=torch.Generator().manual_seed(SEED+4))[:2000 if QUICK_RUN else len(train_x)]
paired_train=train_x[paired_ids]
assert PD(torch.cat([edges(examples),examples],1).to(device)).shape==(8,1,7,7)
panel([edges(examples),examples],["Edge condition","Paired target"],"paired_data.png","Paired translation: same digit in both rows")
paired_history=[]; best_val=float("inf"); best_p=None; best_p_epoch=0
for epoch in range(1,(4 if QUICK_RUN else 20)+1):
    totals=np.zeros(3);count=0
    for bi,(y,) in enumerate(loader(paired_train,seed=SEED+epoch)):
        y=y.to(device); x=edges(y); audit=epoch==1 and bi==0
        if audit: p0=snapshot(P)
        frozen(PD,False);pd_opt.zero_grad(set_to_none=True)
        fake=P(x)
        ld=real_loss(PD(torch.cat([x,y],1)))+fake_loss(PD(torch.cat([x,fake.detach()],1)))
        finite(ld);ld.backward();pd_opt.step()
        if audit: assert unchanged(P,p0)
        frozen(PD);pd_opt.zero_grad(set_to_none=True);pg.zero_grad(set_to_none=True)
        if audit: d0=snapshot(PD)
        adv=real_loss(PD(torch.cat([x,fake],1)));l1=F.l1_loss(fake,y);lg=adv+50*l1
        finite(lg);lg.backward();pg.step()
        if audit:
            assert unchanged(PD,d0) and not unchanged(P,p0)
            checks["paired_update_isolation"]=True
        frozen(PD,False);totals+=np.array([ld.item(),adv.item(),l1.item()])*len(y);count+=len(y)
    P.eval()
    with torch.no_grad(): vl=F.l1_loss(P(edges(val_x).to(device)),val_x.to(device)).item()
    if vl<best_val: best_val,best_p,best_p_epoch=vl,copy.deepcopy(P.state_dict()),epoch
    paired_history.append({"epoch":epoch,"D":totals[0]/count,"G_adversarial":totals[1]/count,"train_L1":totals[2]/count,"val_L1":vl})
    print("Paired",paired_history[-1]);P.train()
P.load_state_dict(best_p);P.eval();PD.eval()

In [ ]:
with torch.no_grad():
    paired_pred=P(edges(test_x).to(device)).cpu()
    patch_scores=PD(torch.cat([edges(examples).to(device),P(edges(examples).to(device))],1)).sigmoid().cpu()
paired_metrics={"best_epoch":best_p_epoch,"validation_L1":best_val,
    "test_L1":F.l1_loss(paired_pred,test_x).item(),
    "edge_identity_test_L1":F.l1_loss(edges(test_x),test_x).item(),
    "training_mean_test_L1":F.l1_loss(paired_train.mean(0,keepdim=True).expand_as(test_x),test_x).item()}
print(json.dumps(paired_metrics,indent=2))
panel([edges(examples),paired_pred[view_ids],examples],["Edges","Prediction","True target"],"paired_results.png","Held-out paired translation: fixed examples")
fig,axes=plt.subplots(1,2,figsize=(10,3))
h=pd.DataFrame(paired_history)
axes[0].plot(h.epoch,h.train_L1,label="Train");axes[0].plot(h.epoch,h.val_L1,label="Validation")
axes[0].set(xlabel="Epoch",ylabel="Mean L1 on [-1,1]");axes[0].legend()
im=axes[1].imshow(patch_scores[0,0],vmin=0,vmax=1,cmap="viridis")
axes[1].set(title="D scores: first generated pair",xlabel="Patch column",ylabel="Patch row")
fig.colorbar(im,ax=axes[1]);finish(fig,"paired_diagnostics.png")

## 7. Unpaired translation: a small CycleGAN-style model

Domain X contains ordinary digits; domain Y contains contrast-inverted **different** digits. Split training indices into disjoint X/Y pools and independently permute them each epoch. No aligned target enters the loss. Both pools contain the same digit classes, but not the same source rows.

Two generators Gxy and Gyx and two domain discriminators are trained. We use least-squares adversarial terms: `(D(real)−1)²`, `D(fake)²`, and `(D(G(input))−1)²`. Each discriminator averages real/fake terms with a factor ½. Generator total = adversarial (both directions) + 10 × cycle L1 (both directions) + 5 × identity L1 (both directions).

Cycles are `Gyx(Gxy(x))≈x` and `Gxy(Gyx(y))≈y`; identities are `Gxy(y)≈y` and `Gyx(x)≈x`. Identity and cycle losses constrain mappings, but do not prove the correct mapping. Our synthetic contrast inversion gives a known target **only for final evaluation**; a model may preserve input contrast while obtaining a low cycle loss. We do not use test targets to tune or select it. Narrow U-Nets and no replay buffer make this an objective/update demonstration, not the original CycleGAN implementation.

In [ ]:
pool_ids=torch.randperm(len(train_x),generator=torch.Generator().manual_seed(SEED+5))
size=1000 if QUICK_RUN else len(train_x)//2
x_ids,y_ids=pool_ids[:size],pool_ids[size:2*size]
assert not set(x_ids.tolist()) & set(y_ids.tolist())
unpaired_x,unpaired_y=train_x[x_ids],-train_x[y_ids]
panel([unpaired_x[:8],unpaired_y[:8]],["Domain X","Unpaired domain Y"],"unpaired_data.png","Different digits: columns are not paired targets")
Gxy,Gyx=UNet().to(device),UNet().to(device)
Dx,Dy=PatchD(1).to(device),PatchD(1).to(device)
cg=torch.optim.Adam(itertools.chain(Gxy.parameters(),Gyx.parameters()),lr=.001,betas=(.5,.999))
cd=torch.optim.Adam(itertools.chain(Dx.parameters(),Dy.parameters()),lr=.0002,betas=(.5,.999))
cycle_history=[];CYCLE_EPOCHS=4 if QUICK_RUN else 25
for epoch in range(1,CYCLE_EPOCHS+1):
    totals=np.zeros(4);count=0
    for bi,((x,),(y,)) in enumerate(zip(loader(unpaired_x,seed=100+epoch),loader(unpaired_y,seed=200+epoch))):
        x,y=x.to(device),y.to(device);audit=epoch==1 and bi==0
        frozen(Dx);frozen(Dy);cd.zero_grad(set_to_none=True);cg.zero_grad(set_to_none=True)
        if audit: dx0,dy0,gx0,gy0=snapshot(Dx),snapshot(Dy),snapshot(Gxy),snapshot(Gyx)
        fy,fx=Gxy(x),Gyx(y)
        adv=((Dy(fy)-1)**2).mean()+((Dx(fx)-1)**2).mean()
        cyc=F.l1_loss(Gyx(fy),x)+F.l1_loss(Gxy(fx),y)
        ident=F.l1_loss(Gxy(y),y)+F.l1_loss(Gyx(x),x)
        lg=adv+10*cyc+5*ident;finite(lg);lg.backward();cg.step()
        if audit:
            assert unchanged(Dx,dx0) and unchanged(Dy,dy0)
            assert not unchanged(Gxy,gx0) and not unchanged(Gyx,gy0)
            gx1,gy1=snapshot(Gxy),snapshot(Gyx)
        frozen(Dx,False);frozen(Dy,False);cd.zero_grad(set_to_none=True)
        ld=.5*(((Dx(x)-1)**2).mean()+(Dx(fx.detach())**2).mean()+((Dy(y)-1)**2).mean()+(Dy(fy.detach())**2).mean())
        finite(ld);ld.backward();cd.step()
        if audit:
            assert unchanged(Gxy,gx1) and unchanged(Gyx,gy1)
            assert not unchanged(Dx,dx0) and not unchanged(Dy,dy0)
            checks["cycle_update_isolation"]=True
        totals+=np.array([adv.item(),cyc.item(),ident.item(),ld.item()])*len(x);count+=len(x)
    cycle_history.append(dict(zip(["adversarial","cycle_L1_sum","identity_L1_sum","D_sum"],(totals/count).tolist()),epoch=epoch))
    print("Unpaired",cycle_history[-1])
for model in (Gxy,Gyx,Dx,Dy):model.eval()
checks["unpaired_disjoint_source_rows"]=True

In [ ]:
with torch.no_grad():
    cx=test_x.to(device);cy=-cx
    fy=Gxy(cx);fx=Gyx(cy)
    cycle_metrics={"X_to_Y_known_target_L1":F.l1_loss(fy,cy).item(),
        "Y_to_X_known_target_L1":F.l1_loss(fx,cx).item(),
        "X_cycle_L1":F.l1_loss(Gyx(fy),cx).item(),"Y_cycle_L1":F.l1_loss(Gxy(fx),cy).item(),
        "identity_baseline_target_L1":F.l1_loss(cx,cy).item(),"analytic_inversion_target_L1":0.0}
    cycled=Gyx(fy).cpu();fy=fy.cpu()
print(json.dumps(cycle_metrics,indent=2))
panel([examples,fy[view_ids],cycled[view_ids],-examples],
      ["Input X","Learned Y","Cycle back X","Known target Y"],"cycle_results.png","Low cycle error need not imply the intended translation")
fig,axes=plt.subplots(1,2,figsize=(10,3));h=pd.DataFrame(cycle_history)
for key in ["cycle_L1_sum","identity_L1_sum"]:axes[0].plot(h.epoch,h[key],label=key)
for key in ["adversarial","D_sum"]:axes[1].plot(h.epoch,h[key],label=key)
for ax in axes:ax.set_xlabel("Epoch");ax.legend()
axes[0].set_ylabel("Unweighted L1 sums");axes[1].set_ylabel("Least-squares loss sums")
finish(fig,"cycle_losses.png")

## 8. Save portable checkpoints and verify reloads

Save all eight networks, configuration, fixed noise, histories, and diagnostics. Reload on CPU and compare outputs for fixed inputs. These are **inference checkpoints**, not exact training-resume snapshots: optimizer and RNG states are not included. GAN/CycleGAN checkpoints are final epoch; the paired generator is validation-selected, whereas its discriminator is final epoch. The latter pair must not be described as a resumable training state.

In [ ]:
specs={"gan_G":(G,Generator,fixed_z),"gan_D":(D,Discriminator,examples.to(device)),
       "paired_G":(P,UNet,edges(examples).to(device)),"paired_D":(PD,PatchD,torch.cat([edges(examples),examples],1).to(device)),
       "cycle_Gxy":(Gxy,UNet,examples.to(device)),"cycle_Gyx":(Gyx,UNet,-examples.to(device)),
       "cycle_Dx":(Dx,lambda:PatchD(1),examples.to(device)),"cycle_Dy":(Dy,lambda:PatchD(1),-examples.to(device))}
reload_checks={}
for name,(model,factory,input_tensor) in specs.items():
    model.eval()
    torch.save({"state_dict":{k:v.detach().cpu() for k,v in model.state_dict().items()},
                "seed":SEED,"quick_run":QUICK_RUN},OUTPUT_DIR/f"{name}.pt")
    restored=factory().cpu().eval()
    restored.load_state_dict(torch.load(OUTPUT_DIR/f"{name}.pt",map_location="cpu",weights_only=True)["state_dict"])
    with torch.no_grad():
        expected=model(input_tensor).cpu();actual=restored(input_tensor.cpu())
    torch.testing.assert_close(expected,actual,rtol=1e-4,atol=1e-5)
    reload_checks[name]=True
pd.DataFrame(gan_history).to_csv(OUTPUT_DIR/"gan_history.csv",index=False)
pd.DataFrame(paired_history).to_csv(OUTPUT_DIR/"paired_history.csv",index=False)
pd.DataFrame(cycle_history).to_csv(OUTPUT_DIR/"cycle_history.csv",index=False)
assert len(pd.read_csv(OUTPUT_DIR/"gan_history.csv"))==GAN_EPOCHS
checks["csv_roundtrip"]=True
torch.save({"fixed_z":fixed_z.cpu(),"inversion_best_z":best_z.cpu()},OUTPUT_DIR/"latent_codes.pt")
metrics={"environment":versions,"splits":[len(train_x),len(val_x),len(test_x)],
         "gan_epochs":GAN_EPOCHS,"gan_history":gan_history,"gif_frames":len(frames),
         "inversion":{"initial_MSE":inversion[0],"best_MSE":best_mse},
         "paired":paired_metrics,"paired_training_images":len(paired_train),
         "cycle":cycle_metrics,"cycle_epochs":CYCLE_EPOCHS,"unpaired_images_per_domain":size,
         "checkpoint_reload":reload_checks,"checks":checks,"figures":FIGURES,
         "seconds_after_imports":time.perf_counter()-run_started}
(OUTPUT_DIR/"metrics.json").write_text(json.dumps(metrics,indent=2)+"\n")
print("TUTORIAL_11_COMPLETE")
print(json.dumps(metrics,indent=2))

## Observed quick-run behavior (Colab CPU, 2026-10-12)

The twelve-epoch GAN produced varied but still fragmented digit-like strokes. The fixed-noise grid is not a demonstration of high-quality or complete digit coverage. Inversion of the fixed target reduced MSE from 0.3515 to 0.1159 without changing generator parameters.

Paired translation selected epoch 4 and reached test L1 0.1017 versus 0.3083 for returning the edge input and 0.3009 for the training mean. This does not isolate the adversarial term's benefit; the discriminator's nearly uniform patch scores are not a quality certificate.

The unpaired model did **not** learn clean contrast inversion in four epochs: X→Y target L1 was 1.2543 while X cycle L1 was only 0.1813. Visible checkerboard artifacts and incorrect contrast remain. This is a useful counterexample to judging translation by cycle consistency alone, not a successful CycleGAN benchmark. See the validation report for the exact environment and untested modes.

## Interpretation and exercises

1. Compare initialization, intermediate, and final fixed-noise frames. Are samples diverse? Do they look like digits? Loss curves alone cannot answer these questions; no FID or class-coverage metric was measured here.
2. For inversion, compare the target, initial output, and fitted output. Repeat on several predeclared test examples without updating G. Does a lower pixel error imply perceptual similarity?
3. Retrain paired translation with adversarial weight zero using the same initialization and schedule. Does adversarial training help relative to L1 alone? Do not choose hyperparameters on the test set.
4. Compare unpaired cycle error against known-target error. The analytic contrast-inversion rule has zero target error by construction. A neural translator is unnecessary for this synthetic task; its purpose is to expose objective ambiguity and optimization mechanics.
5. Larger experiments: set `QUICK_RUN=False` before running from the top. This still uses MNIST and the teaching architectures, not the original full-resolution facades or horse/zebra benchmark. Consult the original projects below for those experiments and their data/compute requirements.

## Troubleshooting and next steps

- Run cells from the top after a runtime restart; a saved notebook does not preserve Python variables.
- If a data download fails, check connectivity and rerun that cell. If imports fail, use a fresh runtime.
- Download files from `outputs/tutorial_11/` using Colab’s Files sidebar before disconnecting.
- Change one experimental setting at a time and record what you changed.

## Attribution

License information is available in the repository `LICENSE`.

Sources: [GAN paper](https://arxiv.org/abs/1406.2661), [DCGAN paper](https://arxiv.org/abs/1511.06434), [pix2pix paper and project](https://phillipi.github.io/pix2pix/), [CycleGAN paper and project](https://junyanz.github.io/CycleGAN/), and [MNIST via torchvision](https://docs.pytorch.org/vision/stable/generated/torchvision.datasets.MNIST.html). Data are downloaded at runtime; the repository code license does not replace dataset terms.